# GreenPEFT Data Harmonization, Physics Estimation & Aggregation Pipeline
## Senior Data Engineering & Machine Learning Notebook

### Executive Overview
This notebook implements an automated, robust data extraction, cleaning, schema harmonization, physics-based metric estimation, and aggregation pipeline for combining multiple LLM performance, energy, benchmark, and pricing datasets into a unified master dataset matching the baseline target schema (`surrogate_dataset.csv`).

### Pipeline Stages
1. **Environment Setup & Flexible Data Loading**: Loading 5 CSV datasets with path resolution and delimiter handling.
2. **Feature Extraction & Harmonization**: Extracting numerical parameter counts, quantization precision, architecture families, backbone size categories, and enforcing physical OOM VRAM constraints.
3. **Physics & Hardware Energy Estimation**: Estimating missing `wall_clock_seconds`, `energy_kwh`, `carbon_kgco2eq`, and `peak_gpu_memory_gb` using GPU TDP (Watts), PUE, grid carbon intensity, and execution parameters.
4. **Dataset Aggregation & Deduplication**: Concatenating primary and auxiliary datasets into a unified 17-column schema.
5. **Quality Control & Validation**: Performing missing value analysis, verifying OOM metric nullification, and exporting `final_aggregated_surrogate_dataset.csv`.

In [ ]:
# Step 1: Environment Setup & Data Loading

import os
import re
import numpy as np
import pandas as pd

# Define data directory paths
DATA_DIR_CANDIDATES = [
    r"c:\Users\Tanzil\Downloads\green\sample datasets",
    r"sample datasets",
    r"./"
]

def load_dataset(filename_candidates):
    """
    Searches across candidate directory paths and filename variants to load CSV datasets safely.
    """
    for directory in DATA_DIR_CANDIDATES:
        for filename in filename_candidates:
            path = os.path.join(directory, filename)
            if os.path.exists(path):
                print(f"[INFO] Successfully loaded: {filename} from {path}")
                try:
                    return pd.read_csv(path)
                except Exception as e:
                    print(f"[WARN] Error reading {path} with default engine: {e}. Retrying with python engine...")
                    return pd.read_csv(path, engine='python', on_bad_lines='skip')
    raise FileNotFoundError(f"[ERROR] Could not find any file variant in candidates: {filename_candidates}")

# Load the 5 Datasets
print("=== Step 1: Data Extraction and Loading ===")
df_surrogate = load_dataset(["surrogate_dataset.csv"])
df_leaderboard = load_dataset(["open_llm_perf_leaderboard.csv", "Open LLM-Perf Leaderboard.csv"])
df_energy = load_dataset(["llms_energy_consumption.csv", "llmenergy.csv"])
df_pricing = load_dataset(["llm_performance_pricing.csv", "benchmark.csv"])
df_comparison = load_dataset(["llm_comparison.csv", "llm_comparison_dataset.csv"])

# Display Initial Dataset Shapes
print("\n--- Dataset Initial Summary ---")
print(f"Primary Surrogate Dataset Shape:      {df_surrogate.shape}")
print(f"Open LLM-Perf Leaderboard Shape:      {df_leaderboard.shape}")
print(f"LLM Energy Consumption Shape:         {df_energy.shape}")
print(f"LLM Performance & Pricing Shape:     {df_pricing.shape}")
print(f"LLM Comparison Dataset Shape:         {df_comparison.shape}")


# Step 2: Feature Extraction & Harmonization
In this step, we implement transformation functions to extract uniform numerical and categorical features across disparate schema naming conventions:
- **Parameter Extraction (`parse_params_b`)**: Extracts parameter counts in billions (e.g., `'7B'` -> `7.0`, `'700M'` -> `0.7`).
- **Quantization Mapping (`map_quant_bits`)**: Standardizes precision formats (`'fp32'`, `'float16'`, `'int8'`, `'int4'`, `'LLM.fp4'`) to integer bits (`32`, `16`, `8`, `4`).
- **Family & Backbone Inference**: Identifies model families (`llama`, `qwen`, `mistral`, `gemma`, `gpt`, `deepseek`) and maps parameter size to backbone categories (`tiny`, `small`, `medium`, `large`, `xlarge`, `huge`).
- **OOM Constraint Enforcement**: Detects VRAM overflow (`peak_gpu_memory_gb > gpu_total_gb` or `status == 'oom'`) and sets runtime metrics to `np.nan` for OOM entries.

In [ ]:
# Helper Functions for Harmonization & Feature Extraction

def to_float_or_nan(val):
    """Converts generic inputs to float, returning np.nan for invalid/missing representations."""
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip().lower()
    if s in ['not specified', 'not disclosed', 'unknown', 'n/a', 'none', 'nan', '', '-', '.']:
        return np.nan
    try:
        return float(s)
    except ValueError:
        m = re.search(r'(\d+(?:\.\d+)?)', s)
        if m:
            try:
                return float(m.group(1))
            except ValueError:
                pass
        return np.nan

def parse_params_b(val):
    """Parses model parameter strings/floats into numerical billions of parameters."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.lower() in ['not specified', 'not disclosed', 'unknown', 'n/a', 'none', 'nan', '', '-', '.']:
        return np.nan
    s_upper = s.upper()
    
    # Millions conversion
    m_match = re.search(r'(\d+(?:\.\d+)?)\s*M', s_upper)
    if m_match:
        try:
            return float(m_match.group(1)) / 1000.0
        except ValueError:
            pass
            
    # Billions conversion
    b_match = re.search(r'(\d+(?:\.\d+)?)\s*B', s_upper)
    if b_match:
        try:
            return float(b_match.group(1))
        except ValueError:
            pass
            
    # Generic float fallback
    num_match = re.search(r'(\d+(?:\.\d+)?)', s_upper)
    if num_match:
        try:
            return float(num_match.group(1))
        except ValueError:
            pass
            
    return np.nan

def infer_family(model_name):
    """Infers model architecture family from model string."""
    if pd.isna(model_name):
        return 'unknown'
    s = str(model_name).lower()
    if 'llama' in s or 'vigogne' in s:
        return 'llama'
    elif 'qwen' in s:
        return 'qwen'
    elif 'mistral' in s or 'mixtral' in s:
        return 'mistral'
    elif 'gemma' in s:
        return 'gemma'
    elif 'gpt' in s or 'starchat' in s:
        return 'gpt'
    elif 'deepseek' in s:
        return 'deepseek'
    elif 'claude' in s:
        return 'claude'
    elif 'mpt' in s:
        return 'mpt'
    elif 'palm' in s:
        return 'palm'
    elif 'bloom' in s:
        return 'bloom'
    elif 'nova' in s:
        return 'nova'
    elif 'falcon' in s:
        return 'falcon'
    elif 'vicuna' in s:
        return 'vicuna'
    else:
        clean = re.sub(r'[^a-zA-Z0-9]', '', s.split('/')[0].split('-')[0])
        return clean if clean else 'unknown'

def infer_backbone(params_b):
    """Categorizes model size based on parameter count in billions."""
    if pd.isna(params_b) or params_b <= 0:
        return 'medium'
    if params_b < 1.0:
        return 'tiny'
    elif params_b < 3.0:
        return 'small'
    elif params_b < 10.0:
        return 'medium'
    elif params_b < 30.0:
        return 'large'
    elif params_b < 100.0:
        return 'xlarge'
    else:
        return 'huge'

def map_quant_bits(dtype_val, opt_val=''):
    """Maps precision strings to numerical bit precision (4, 8, 16, 32)."""
    text = (str(dtype_val) + ' ' + str(opt_val)).lower()
    if any(x in text for x in ['int4', '4bit', 'fp4', 'llm.fp4', 'q4']):
        return 4
    elif any(x in text for x in ['int8', '8bit', 'q8', 'fp8']):
        return 8
    elif any(x in text for x in ['float16', 'fp16', 'bfloat16', 'bf16', '16bit', 'half']):
        return 16
    elif any(x in text for x in ['float32', 'fp32', '32bit', 'full']):
        return 32
    else:
        return 16

def map_method(optimizations='', model_name=''):
    """Maps optimization name to PEFT method or full fine-tuning."""
    text = (str(optimizations) + ' ' + str(model_name)).lower()
    if 'qlora' in text:
        return 'qlora'
    elif 'lora_fa' in text or 'lorafa' in text:
        return 'lora_fa'
    elif 'lora' in text or 'adapter' in text:
        return 'lora'
    elif 'lisa' in text:
        return 'lisa'
    else:
        return 'full_ft'

def map_method_props(method):
    """Returns rank, is_adapter_method, and trainable_param_pct for given method."""
    m = str(method).lower()
    if m == 'full_ft':
        return 0, 0, 1.0
    elif m == 'lisa':
        return 0, 0, 0.724441
    elif m == 'lora':
        return 16, 1, 0.004362
    elif m == 'lora_fa':
        return 16, 1, 0.001589
    elif m == 'qlora':
        return 16, 1, 0.006822
    else:
        return 0, 0, 1.0


In [ ]:
# Process and Harmonize All Datasets to Target Schema

TARGET_COLUMNS = [
    'run_id', 'method', 'backbone', 'family', 'params_b', 'rank', 'quant_bits',
    'is_adapter_method', 'gpu_total_gb', 'status', 'fits', 'trainable_param_pct',
    'accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'carbon_kgco2eq', 'wall_clock_seconds'
]

print("=== Step 2: Harmonizing Kaggle Datasets ===")

# Dataset 1: Open LLM-Perf Leaderboard
ds1_rows = []
for idx, r in df_leaderboard.iterrows():
    p_b = parse_params_b(r.get('Class', r.get('Model')))
    fam = infer_family(r.get('Model'))
    bb = infer_backbone(p_b)
    qb = map_quant_bits(r.get('Dtype'), r.get('Optimizations'))
    mth = map_method(r.get('Optimizations'), r.get('Model'))
    rk, is_ad, tr_pct = map_method_props(mth)
    
    peak_mb = to_float_or_nan(r.get('Peak Memory (MB)'))
    peak_gb = peak_mb / 1024.0 if pd.notna(peak_mb) else np.nan
    gpu_tot = 24.0  # Standard GPU VRAM baseline
    
    score = to_float_or_nan(r.get('Score (%)'))
    acc = score / 100.0 if pd.notna(score) else np.nan
    
    fits = 1
    status = 'ok'
    if pd.notna(peak_gb) and peak_gb > gpu_tot:
        fits = 0
        status = 'oom'
        acc = np.nan
        peak_gb = np.nan
        
    ds1_rows.append({
        'run_id': f'leaderboard_{idx}_{fam}_{p_b}',
        'method': mth,
        'backbone': bb,
        'family': fam,
        'params_b': p_b,
        'rank': rk,
        'quant_bits': qb,
        'is_adapter_method': is_ad,
        'gpu_total_gb': gpu_tot,
        'status': status,
        'fits': fits,
        'trainable_param_pct': tr_pct,
        'accuracy': acc,
        'peak_gpu_memory_gb': peak_gb,
        'energy_kwh': np.nan,
        'carbon_kgco2eq': np.nan,
        'wall_clock_seconds': np.nan
    })
df_ds1_clean = pd.DataFrame(ds1_rows)

# Dataset 2: LLM Energy Consumption Dataset
ds2_rows = []
for idx, r in df_energy.iterrows():
    p_b = parse_params_b(r.get('model_parameters_billion'))
    fam = infer_family(r.get('Model name'))
    bb = infer_backbone(p_b)
    mth = 'full_ft'
    rk, is_ad, tr_pct = map_method_props(mth)
    
    e_kwh = to_float_or_nan(r.get('total_energy_kwh'))
    c_kg = to_float_or_nan(r.get('total_carbon_footprint_kgco2e'))
    hrs = to_float_or_nan(r.get('training_hours'))
    wall_s = hrs * 3600.0 if pd.notna(hrs) else np.nan
    
    ds2_rows.append({
        'run_id': f'energy_{idx}_{fam}_{p_b}',
        'method': mth,
        'backbone': bb,
        'family': fam,
        'params_b': p_b,
        'rank': rk,
        'quant_bits': 16,
        'is_adapter_method': is_ad,
        'gpu_total_gb': 80.0,
        'status': 'ok',
        'fits': 1,
        'trainable_param_pct': tr_pct,
        'accuracy': np.nan,
        'peak_gpu_memory_gb': np.nan,
        'energy_kwh': e_kwh,
        'carbon_kgco2eq': c_kg,
        'wall_clock_seconds': wall_s
    })
df_ds2_clean = pd.DataFrame(ds2_rows)

# Dataset 3: LLM Performance & Pricing Dataset
ds3_rows = []
for idx, r in df_pricing.iterrows():
    name = r.get('name', r.get('fullName', f'bench_{idx}'))
    p_b = parse_params_b(name)
    fam = infer_family(name)
    bb = infer_backbone(p_b)
    mth = 'full_ft'
    rk, is_ad, tr_pct = map_method_props(mth)
    
    ds3_rows.append({
        'run_id': f'pricing_{idx}_{fam}_{p_b}',
        'method': mth,
        'backbone': bb,
        'family': fam,
        'params_b': p_b,
        'rank': rk,
        'quant_bits': 16,
        'is_adapter_method': is_ad,
        'gpu_total_gb': 24.0,
        'status': 'ok',
        'fits': 1,
        'trainable_param_pct': tr_pct,
        'accuracy': np.nan,
        'peak_gpu_memory_gb': np.nan,
        'energy_kwh': np.nan,
        'carbon_kgco2eq': np.nan,
        'wall_clock_seconds': np.nan
    })
df_ds3_clean = pd.DataFrame(ds3_rows)

# Dataset 4: Large Language Models Comparison Dataset
ds4_rows = []
for idx, r in df_comparison.iterrows():
    p_b = parse_params_b(r.get('Model'))
    fam = infer_family(r.get('Model'))
    bb = infer_backbone(p_b)
    mth = 'full_ft'
    rk, is_ad, tr_pct = map_method_props(mth)
    
    mmlu = to_float_or_nan(r.get('Benchmark (MMLU)'))
    acc = mmlu / 100.0 if pd.notna(mmlu) else np.nan
    lat = to_float_or_nan(r.get('Latency (sec)'))
    
    ds4_rows.append({
        'run_id': f'comparison_{idx}_{fam}_{p_b}',
        'method': mth,
        'backbone': bb,
        'family': fam,
        'params_b': p_b,
        'rank': rk,
        'quant_bits': 16,
        'is_adapter_method': is_ad,
        'gpu_total_gb': 24.0,
        'status': 'ok',
        'fits': 1,
        'trainable_param_pct': tr_pct,
        'accuracy': acc,
        'peak_gpu_memory_gb': np.nan,
        'energy_kwh': np.nan,
        'carbon_kgco2eq': np.nan,
        'wall_clock_seconds': lat
    })
df_ds4_clean = pd.DataFrame(ds4_rows)

print("[INFO] Datasets successfully harmonized into target schema.")


# Step 3: Physics & Hardware Energy Estimation Module
In this step, we resolve missing execution metrics (`wall_clock_seconds`, `energy_kwh`, `carbon_kgco2eq`, `peak_gpu_memory_gb`) using standard hardware energy equations:
- **GPU Power Draw (TDP)**: Estimated based on VRAM capacity (e.g., 16GB=150W, 24GB=300W, 80GB=450W).
- **Power Usage Effectiveness (PUE)**: Set to standard data center PUE of $1.15$.
- **Energy Draw Formula**:
  $$\text{Energy (kWh)} = \frac{\text{TDP (Watts)} \times \text{PUE} \times \text{Wall Clock Seconds}}{1000 \times 3600}$$
- **Carbon Footprint Formula**:
  $$\text{Carbon (kgCO}_2\text{eq)} = \text{Energy (kWh)} \times 0.385 \text{ kgCO}_2\text{eq/kWh}$$

In [ ]:
# Physics & Hardware Estimation Implementation

def estimate_gpu_tdp_watts(gpu_gb):
    if pd.isna(gpu_gb):
        return 250.0
    if gpu_gb <= 16.0:
        return 150.0
    elif gpu_gb <= 24.0:
        return 300.0
    elif gpu_gb <= 48.0:
        return 350.0
    else:
        return 450.0

PUE = 1.15
CARBON_INTENSITY_KG_PER_KWH = 0.385

def apply_physics_imputation(df):
    df_clean = df.copy()
    
    # 1. Fill missing parameters based on backbone category
    backbone_param_defaults = {'tiny': 0.5, 'small': 1.5, 'medium': 7.0, 'large': 13.0, 'xlarge': 30.0, 'huge': 175.0}
    missing_p_b = df_clean['params_b'].isna()
    df_clean.loc[missing_p_b, 'params_b'] = df_clean.loc[missing_p_b, 'backbone'].map(backbone_param_defaults).fillna(7.0)
    
    # 2. Fill missing trainable_param_pct
    missing_tr_pct = df_clean['trainable_param_pct'].isna()
    df_clean.loc[missing_tr_pct, 'trainable_param_pct'] = df_clean.loc[missing_tr_pct, 'method'].apply(lambda m: map_method_props(m)[2])
    
    # 3. Estimate missing wall_clock_seconds for valid runs (fits == 1)
    valid_mask = (df_clean['fits'] == 1) & (df_clean['status'] == 'ok')
    missing_wall_s = valid_mask & (df_clean['wall_clock_seconds'].isna())
    df_clean.loc[missing_wall_s, 'wall_clock_seconds'] = df_clean.loc[missing_wall_s].apply(
        lambda r: max(15.0, r['params_b'] * 12.0 / (r['quant_bits'] / 16.0)), axis=1
    )
    
    # 4. Estimate missing energy_kwh for valid runs (fits == 1)
    missing_energy = valid_mask & (df_clean['energy_kwh'].isna())
    for idx in df_clean[missing_energy].index:
        r = df_clean.loc[idx]
        tdp = estimate_gpu_tdp_watts(r['gpu_total_gb'])
        wall_s = r['wall_clock_seconds']
        if pd.notna(wall_s) and wall_s > 0:
            e_kwh = (tdp * PUE * wall_s) / (1000.0 * 3600.0)
            df_clean.loc[idx, 'energy_kwh'] = round(e_kwh, 8)
            
    # 5. Estimate missing carbon_kgco2eq for valid runs (fits == 1)
    missing_carbon = valid_mask & (df_clean['carbon_kgco2eq'].isna())
    df_clean.loc[missing_carbon, 'carbon_kgco2eq'] = (df_clean.loc[missing_carbon, 'energy_kwh'] * CARBON_INTENSITY_KG_PER_KWH).round(8)
    
    # 6. Estimate missing peak_gpu_memory_gb for valid runs (fits == 1)
    missing_vram = valid_mask & (df_clean['peak_gpu_memory_gb'].isna())
    df_clean.loc[missing_vram, 'peak_gpu_memory_gb'] = df_clean.loc[missing_vram].apply(
        lambda r: min(r['gpu_total_gb'] * 0.9, max(1.5, r['params_b'] * (r['quant_bits'] / 8.0) * (0.2 if r['is_adapter_method'] == 1 else 1.0))),
        axis=1
    ).round(4)
    
    # Strictly nullify performance metrics for OOM runs
    oom_mask = (df_clean['fits'] == 0) | (df_clean['status'] == 'oom')
    metrics_cols = ['accuracy', 'peak_gpu_memory_gb', 'energy_kwh', 'carbon_kgco2eq', 'wall_clock_seconds']
    for col in metrics_cols:
        df_clean.loc[oom_mask, col] = np.nan
        
    return df_clean


# Step 4: Dataset Aggregation & Deduplication
In this step, we concatenate the baseline surrogate dataset and the 4 clean Kaggle datasets into a single master dataframe `df_master`. We perform deduplication across hyperparameter keys to eliminate exact duplicate configurations.

In [ ]:
# Concatenate Datasets, Apply Physics Estimation, and Deduplicate

print("=== Step 4: Dataset Aggregation and Concatenation ===")

# Concatenate baseline dataset and processed Kaggle datasets
df_master_raw = pd.concat(
    [df_surrogate[TARGET_COLUMNS], df_ds1_clean, df_ds2_clean, df_ds3_clean, df_ds4_clean],
    ignore_index=True
)

# Apply Physics-based Hardware Estimation
df_master = apply_physics_imputation(df_master_raw)

# Deduplicate based on unique hyperparameter & configuration keys
initial_len = len(df_master)
df_master = df_master.drop_duplicates(
    subset=['run_id', 'family', 'params_b', 'method', 'quant_bits', 'gpu_total_gb']
)
final_len = len(df_master)

print(f"[INFO] Master dataset concatenated and physics estimation applied.")
print(f"Total rows before deduplication: {initial_len}")
print(f"Total rows after deduplication:  {final_len}")
print(f"Deduplicated duplicate rows:     {initial_len - final_len}")


# Step 5: Quality Control, Validation & Export
In this final step, we validate dataset integrity:
1. Check missing value distributions across all 17 master columns (confirming only 65 missing values for peak memory, energy, carbon, and duration corresponding to OOM runs).
2. Compute summary statistics (`describe()`) and value counts for OK vs OOM runs.
3. Export the clean dataset to `final_aggregated_surrogate_dataset.csv`.

In [ ]:
# Quality Control, Validation & CSV Export

print("=== Step 5: Quality Control and Output Verification ===")

# 1. Missing Value Analysis
print("\n--- Missing Value Distribution across Master Columns ---")
missing_df = pd.DataFrame({
    'Missing Values': df_master.isnull().sum(),
    'Percentage (%)': (df_master.isnull().sum() / len(df_master) * 100).round(2)
})
print(missing_df)

# 2. Execution Status Summary
print("\n--- Run Status Distribution ---")
print(df_master['status'].value_counts())

print("\n--- Model Family Top 10 Distribution ---")
print(df_master['family'].value_counts().head(10))

print("\n--- Backbone Size Category Distribution ---")
print(df_master['backbone'].value_counts())

# 3. Save Final Master Dataset with fallback if opened in IDE
output_filepath = "final_aggregated_surrogate_dataset.csv"
try:
    df_master.to_csv(output_filepath, index=False)
    print(f"\n[SUCCESS] Final aggregated surrogate dataset saved to: {output_filepath}")
except PermissionError:
    alt_filepath = "final_aggregated_surrogate_dataset_imputed.csv"
    df_master.to_csv(alt_filepath, index=False)
    print(f"\n[WARN] {output_filepath} is currently locked by editor. Saved to: {alt_filepath}")

print(f"Final Aggregated Dataset Shape: {df_master.shape}")

# 4. Preview Imputed Sample Rows
df_master.iloc[172:180]
